# Dashboard — Crimen en Los Ángeles: ¿dónde, cuándo y qué ocurre?
## Examen final · Hito 1 — versión beta 2.1.0
**Integrantes:** Felipe Farías · Benjamín Godoy — Visualización de la Información, UDD 2026

**Objetivo:** transformar el análisis del Certamen 2 en un **dashboard interactivo** con Streamlit y Plotly, publicado en Streamlit Community Cloud.

**Plan (misma estructura que el tutorial de tips):**
1. Cargar y limpiar los datos con el código del Certamen 2.
2. Guardar el dataset limpio que va en la carpeta `data` del repositorio.
3. Construir cada gráfico por separado con Plotly Express (`fig.show()`).
4. Traducir cada pieza a Streamlit (widgets, `st.metric`, `st.plotly_chart`).
5. Ensamblar todo en `crimen_app.py` y crear los archivos del repositorio.
6. Probar la app aquí en Colab y publicarla (GitHub → Streamlit Community Cloud).

> **Pregunta que responde el dashboard:** ¿dónde, cuándo y qué delitos ocurren en Los Ángeles (2016–2019)?
> **Dirigido a:** equipos de planificación de patrullaje y prevención de LAPD.

## 0. Instalar librerías

In [ ]:
!pip install streamlit plotly pandas -q

## 1. Cargar los datos (código del Certamen 2)
Subir a Colab el archivo `Crime_Data_from_2010_to_2019.csv` (o el `.zip`). Se filtran los años 2016–2019, igual que en el Certamen 2.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import os
import zipfile

#  descomprimir el archivo .zip
if os.path.exists("Crime_Data_from_2010_to_2019.csv.zip") and not os.path.exists("Crime_Data_from_2010_to_2019.csv"):
    with zipfile.ZipFile("Crime_Data_from_2010_to_2019.csv.zip", "r") as z:
        z.extractall()

#  carga de datos y filtrado 2016 - 2019
df_crudo = pd.read_csv("Crime_Data_from_2010_to_2019.csv", low_memory=False)
df_crudo.columns = df_crudo.columns.str.strip()
df_crudo["DATE OCC"] = pd.to_datetime(df_crudo["DATE OCC"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
df = df_crudo[df_crudo["DATE OCC"].dt.year.between(2016, 2019)].copy()

print(f"Filas 2016-2019: {len(df)}")

### Limpieza (decisiones del Certamen 2)
| Variable | Decisión |
|---|---|
| `Vict Age` ≤ 0 o > 100 | Nulo (edad imposible / error de captura) |
| Coordenadas (0, 0) o fuera de Los Ángeles | Nulo (se informan en la app) |
| `Vict Sex` 'H', '-' o nulo | 'X' (desconocido) |
| `Weapon Desc` nulo | Sin arma |

In [ ]:
# 1. Aplicar las decisiones de limpieza del Certamen 2
df["Hora"] = df["TIME OCC"] // 100
df.loc[(df["Vict Age"] <= 0) | (df["Vict Age"] > 100), "Vict Age"] = np.nan
fuera = ~(df["LAT"].between(33.70, 34.35) & df["LON"].between(-118.68, -118.15))
df.loc[fuera, ["LAT", "LON"]] = np.nan
df["Vict Sex"] = df["Vict Sex"].replace({"H": "X", "-": "X"}).fillna("X")

print(f"Edades no validas: {df['Vict Age'].isna().sum()}")
print(f"Coordenadas invalidas: {fuera.sum()}")

## 2. Guardar el dataset limpio para el repositorio
Solo se guardan las columnas que usa el dashboard, con nombres en español. El archivo completo (~900 mil filas) supera los **25 MB** que permite GitHub al subir por la web, así que se guarda una **muestra de 200.000 delitos con `random_state=42`** (se informa en el README).

In [ ]:
# 2. Dataset limpio con las columnas del dashboard
r2 = pd.DataFrame({
    "Fecha": df["DATE OCC"].dt.strftime("%Y-%m-%d"),
    "Hora": df["Hora"],
    "Zona": df["AREA NAME"],
    "Delito": df["Crm Cd Desc"],
    "Gravedad": df["Part 1-2"],
    "Edad": df["Vict Age"],
    "Sexo": df["Vict Sex"],
    "Arma": df["Weapon Desc"].notna().astype(int),
    "Arresto": df["Status Desc"].astype(str).str.contains("Arrest").astype(int),
    "LAT": df["LAT"],
    "LON": df["LON"],
})

os.makedirs("data", exist_ok=True)
r2.sample(200000, random_state=42).to_csv("data/crimen_la_limpio.csv", index=False)
print(f"Tamaño del archivo: {os.path.getsize('data/crimen_la_limpio.csv') / 1024**2:.1f} MB")

### Preparar columnas (lo mismo que hará la app)
Igual que en el tutorial de tips: se traducen los códigos a etiquetas en español y se crean columnas de fecha. Esto es **pandas puro**.

In [ ]:
df = pd.read_csv("data/crimen_la_limpio.csv", parse_dates=["Fecha"])
df["Año"] = df["Fecha"].dt.year
df["Mes"] = df["Fecha"].dt.to_period("M").dt.to_timestamp()
df["Día"] = df["Fecha"].dt.dayofweek.map({0: "Lun", 1: "Mar", 2: "Mié", 3: "Jue", 4: "Vie", 5: "Sáb", 6: "Dom"})
df["Gravedad"] = df["Gravedad"].map({1: "Grave", 2: "Otros"})
df["Sexo"] = df["Sexo"].map({"F": "Mujer", "M": "Hombre"}).fillna("Desconocido")
df["Arma"] = df["Arma"].map({1: "Con arma", 0: "Sin arma"})
df["Arresto"] = df["Arresto"].map({1: "Sí", 0: "No"})

ORDEN_DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
df.head()

## 3. ¿Por qué `@st.cache_data`?
En Streamlit **todo el script se vuelve a ejecutar** cada vez que el usuario mueve un filtro. Leer 200 mil filas en cada clic sería lento. Por eso la carga va dentro de una función con `@st.cache_data`: se ejecuta **una sola vez** y después se reutiliza el resultado.

```python
@st.cache_data
def cargar_datos():
    df = pd.read_csv("data/crimen_la_limpio.csv", parse_dates=["Fecha"])
    # ... transformaciones ...
    return df
```

## 4. Gráfico 1 — Top 10 tipos de delito (univariado)
`value_counts()` para contar y `px.bar` horizontal. `hover_data` personaliza lo que aparece al pasar el cursor.

In [ ]:
COLOR_GRAVEDAD = {"Grave": "#B9544A", "Otros": "#3A3E48"}

r4 = df["Delito"].value_counts().head(10).reset_index()
r4.columns = ["Delito", "Delitos"]
r4["%"] = (r4["Delitos"] / len(df) * 100).round(1)

fig = px.bar(r4, x="Delitos", y="Delito", orientation="h", hover_data={"%": True},
             title="Top 10 tipos de delito", labels={"Delitos": "Cantidad de delitos", "Delito": "Tipo de delito"},
             color_discrete_sequence=["#B9544A"])
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 5. Gráfico 2 — Edad de las víctimas (univariado)
`px.histogram` con `color` para comparar mujeres y hombres (`barmode="overlay"`).

In [ ]:
COLOR_SEXO = {"Mujer": "#D4877A", "Hombre": "#3A3E48", "Desconocido": "#E8C79A"}

fig = px.histogram(df[df["Sexo"] != "Desconocido"], x="Edad", color="Sexo", nbins=40, opacity=0.7,
                   barmode="overlay", color_discrete_map=COLOR_SEXO,
                   title="Edad de las víctimas", labels={"Edad": "Edad de la víctima (años)", "count": "Cantidad"})
fig.update_layout(yaxis_title="Cantidad de víctimas")
fig.show()

## 6. Gráfico 3 — Delitos por mes (temporal)
`groupby().size()` por mes y `px.line`.

In [ ]:
r6 = df.groupby(["Mes", "Gravedad"]).size().reset_index(name="Delitos")

fig = px.line(r6, x="Mes", y="Delitos", color="Gravedad", markers=True, color_discrete_map=COLOR_GRAVEDAD,
              title="Delitos por mes", labels={"Mes": "Mes", "Delitos": "Cantidad de delitos"})
fig.show()

## 7. Gráfico 4 — Día de la semana × hora (temporal)
`px.density_heatmap` cuenta los delitos de cada combinación día–hora.

In [ ]:
fig = px.density_heatmap(df, x="Hora", y="Día", nbinsx=24, color_continuous_scale="YlOrRd",
                         category_orders={"Día": ORDEN_DIAS},
                         title="Delitos por día de la semana y hora",
                         labels={"Hora": "Hora del día (0-23 h)", "Día": "Día de la semana"})
fig.update_layout(coloraxis_colorbar_title="Delitos")
fig.show()

## 8. Gráfico 5 — Arresto con y sin arma (relaciones)
`groupby().mean()` sobre una columna Sí/No da el porcentaje de arrestos.

In [ ]:
r8 = (df.groupby("Arma")["Arresto"].apply(lambda s: (s == "Sí").mean() * 100)).round(1).reset_index(name="% arresto")

fig = px.bar(r8, x="Arma", y="% arresto", color="Arma", text="% arresto",
             color_discrete_map={"Con arma": "#B9544A", "Sin arma": "#E8C79A"},
             title="Delitos que terminan en arresto", labels={"Arma": "Uso de arma", "% arresto": "Delitos con arresto (%)"})
fig.show()

## 9. Mapas — puntos y densidad (componente avanzado)
- `px.scatter_map`: cada punto es un delito. Con más de 100.000 puntos se usa una **muestra con `random_state`**.
- `px.density_map`: concentración de delitos.
- `map_style="open-street-map"` pone el mapa de calles de fondo; `center` y `zoom` fijan la vista inicial en Los Ángeles.

In [ ]:
df_mapa = df.dropna(subset=["LAT", "LON"])
print(f"Delitos sin coordenadas validas: {len(df) - len(df_mapa)}")
muestra = df_mapa.sample(min(20000, len(df_mapa)), random_state=42)
CENTRO = {"lat": 34.05, "lon": -118.35}

fig = px.scatter_map(muestra, lat="LAT", lon="LON", color="Gravedad", color_discrete_map=COLOR_GRAVEDAD,
                     hover_name="Zona", hover_data={"Delito": True, "Fecha": "|%d-%m-%Y", "Hora": True, "LAT": False, "LON": False},
                     center=CENTRO, zoom=9, opacity=0.5, height=550, title="Mapa de puntos de los delitos")
fig.update_layout(map_style="open-street-map")
fig.show()

fig = px.density_map(muestra, lat="LAT", lon="LON", radius=3, center=CENTRO, zoom=9, height=550,
                     color_continuous_scale="YlOrRd", hover_data={"Zona": True}, title="Mapa de densidad de delitos")
fig.update_layout(map_style="open-street-map", coloraxis_colorbar_title="Densidad")
fig.show()

## 10. Del notebook a Streamlit: la "traducción"
| En el notebook | En Streamlit |
|---|---|
| `fig.show()` | `st.plotly_chart(fig, width="stretch")` (equivale a `use_container_width=True` del tutorial, que Streamlit está eliminando) |
| Filtro escrito en el código, ej. `df[df["Año"] == 2017]` | Valor de un widget, ej. `años = st.slider(...)` |
| `print(...)` / tabla de resumen | `st.metric(...)`, `st.dataframe(...)` |
| Una celda = un gráfico | Una sección del script, ordenada con `st.tabs()` y `st.columns()` |

**Widgets que usa la app:**
- **Filtros globales (panel lateral):** `st.slider` (años), `st.multiselect` (zonas), `st.radio` (gravedad).
- **Control local:** `st.selectbox("Colorear por")` dentro de la pestaña de mapas: afecta solo al mapa de puntos.
- `st.metric(..., delta=...)` para los KPIs, `st.expander` para la tabla y `st.stop()` si el filtro deja la tabla vacía.

## 11. Ensamblar la app completa
Igual que `tips_app.py`: configuración, paleta, carga cacheada, filtros, KPIs, gráficos y tabla, cada sección numerada y comentada.

In [ ]:
%%writefile crimen_app.py
# crimen_app.py
# ============================================================
#  DASHBOARD — Crimen en Los Ángeles: ¿dónde, cuándo y qué ocurre?
#  Examen final · Hito 1 · versión beta 2.1.0
#  Felipe Farías · Benjamín Godoy — Visualización de la Información, UDD
#  Ejecutar:  streamlit run crimen_app.py
#
#  Misma estructura que tips_app.py: el script se ejecuta de ARRIBA
#  A ABAJO cada vez que el usuario mueve un filtro.
# ============================================================

import pandas as pd
import plotly.express as px
import streamlit as st

# ── 1. CONFIGURACIÓN DE LA PÁGINA ────────────────────────────
st.set_page_config(
    page_title="Crimen en Los Ángeles",
    page_icon="🚓",
    layout="wide",
)

# ── 2. PALETA DE COLORES Y ESTILO COMPARTIDO ─────────────────
# La misma categoría tiene siempre el mismo color en todos los gráficos.
COLOR_GRAVEDAD = {"Grave": "#B9544A", "Otros": "#3A3E48"}
COLOR_SEXO     = {"Mujer": "#D4877A", "Hombre": "#3A3E48", "Desconocido": "#E8C79A"}
COLOR_ARMA     = {"Con arma": "#B9544A", "Sin arma": "#E8C79A"}
COLOR_ARRESTO  = {"Sí": "#B9544A", "No": "#3A3E48"}
COLORES        = {"Gravedad": COLOR_GRAVEDAD, "Sexo": COLOR_SEXO, "Arma": COLOR_ARMA, "Arresto": COLOR_ARRESTO}
ORDEN_DIAS     = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
CENTRO         = {"lat": 34.05, "lon": -118.35}   # vista inicial del mapa

LAYOUT = dict(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter,Arial,sans-serif", size=12),
    margin=dict(t=50, b=40, l=50, r=20),
    legend=dict(orientation="h", y=-0.2),
)

# ── 3. CARGAR Y PREPARAR LOS DATOS ───────────────────────────
# @st.cache_data: lee el CSV UNA sola vez y guarda el resultado.
# Sin esta línea se volvería a leer en cada clic de un filtro.
@st.cache_data
def cargar_datos():
    # Datos limpios del Certamen 2 (muestra de 200.000 delitos, random_state=42)
    df = pd.read_csv("data/crimen_la_limpio.csv", parse_dates=["Fecha"])

    # Columnas de fecha y etiquetas en español (pandas puro)
    df["Año"] = df["Fecha"].dt.year
    df["Mes"] = df["Fecha"].dt.to_period("M").dt.to_timestamp()
    df["Día"] = df["Fecha"].dt.dayofweek.map(dict(enumerate(ORDEN_DIAS)))
    df["Gravedad"] = df["Gravedad"].map({1: "Grave", 2: "Otros"})
    df["Sexo"] = df["Sexo"].map({"F": "Mujer", "M": "Hombre"}).fillna("Desconocido")
    df["Arma"] = df["Arma"].map({1: "Con arma", 0: "Sin arma"})
    df["Arresto"] = df["Arresto"].map({1: "Sí", 0: "No"})
    return df

df = cargar_datos()

# ── 4. SIDEBAR — FILTROS GLOBALES ────────────────────────────
# Todo lo que está dentro de "with st.sidebar:" va al panel lateral.
# Cada widget devuelve el valor elegido y lo guardamos en una variable.
with st.sidebar:
    st.title("🚓 Crimen en LA")
    st.caption("Filtros globales — afectan todos los KPIs y gráficos")

    # slider de rango → tupla (año mínimo, año máximo)
    año_min, año_max = st.slider(
        "Años",
        min_value=int(df["Año"].min()),
        max_value=int(df["Año"].max()),
        value=(int(df["Año"].min()), int(df["Año"].max())),
    )

    # multiselect → varias zonas a la vez
    zonas_todas = sorted(df["Zona"].unique())
    zonas = st.multiselect("Zona policial", options=zonas_todas, default=zonas_todas)

    # radio → una sola opción
    gravedad = st.radio("Gravedad del delito", options=["Todos", "Grave", "Otros"])

    st.divider()
    st.caption(f"Dataset: {len(df):,} delitos (muestra 2016–2019)")
    st.caption("Versión beta 2.1.0")

# ── 5. APLICAR LOS FILTROS (pandas puro) ─────────────────────
mask = df["Año"].between(año_min, año_max) & df["Zona"].isin(zonas)
if gravedad != "Todos":
    mask &= df["Gravedad"] == gravedad

dff = df[mask]  # dff = "dataframe filtrado"

# ── 6. ENCABEZADO Y PREGUNTA DEL DASHBOARD ───────────────────
st.title("Crimen en Los Ángeles: ¿dónde, cuándo y qué ocurre?")
st.markdown("**Pregunta:** ¿en qué zonas, en qué momentos y qué tipos de delito concentran la actividad criminal "
            "en Los Ángeles entre 2016 y 2019? Pensado para quienes planifican el patrullaje y la prevención.")
st.caption(f"Mostrando {len(dff):,} de {len(df):,} delitos con los filtros aplicados")

# Si el filtro deja la tabla vacía, avisamos y detenemos la app
if dff.empty:
    st.warning("No hay delitos para los filtros seleccionados. Ajusta el panel lateral.")
    st.stop()

# ── 7. KPIs (TARJETAS DE MÉTRICAS) ───────────────────────────
k1, k2, k3, k4 = st.columns(4)

pct_arma       = (dff["Arma"] == "Con arma").mean() * 100
pct_arresto    = (dff["Arresto"] == "Sí").mean() * 100
arresto_global = (df["Arresto"] == "Sí").mean() * 100   # referencia SIN filtros

k1.metric("Delitos", f"{len(dff):,}", delta=f"{len(dff) / len(df) * 100:.0f}% del total", delta_color="off")
k2.metric("Edad mediana de la víctima", f"{dff['Edad'].median():.0f} años")
k3.metric("Delitos con arma", f"{pct_arma:.1f}%")
k4.metric("Delitos con arresto", f"{pct_arresto:.1f}%",
          delta=f"{pct_arresto - arresto_global:+.1f} pts vs total")

st.divider()

# ── 8. PESTAÑAS: DÓNDE · CUÁNDO · QUÉ Y A QUIÉNES ────────────
tab1, tab2, tab3 = st.tabs(["📍 Dónde", "🕒 Cuándo", "📊 Qué y a quiénes"])

# ---------- 8.1 DÓNDE: MAPAS ----------
with tab1:
    df_mapa = dff.dropna(subset=["LAT", "LON"])
    st.caption(f"Se excluyen {len(dff) - len(df_mapa):,} delitos con coordenadas inválidas "
               "(0, 0 o fuera de Los Ángeles). Mapas con una muestra de hasta 20.000 puntos (random_state=42).")
    muestra = df_mapa.sample(min(20000, len(df_mapa)), random_state=42)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("Mapa de puntos")
        # CONTROL LOCAL: este selectbox solo afecta a este mapa
        color_por = st.selectbox("Colorear por", ["Gravedad", "Arma", "Arresto", "Sexo"], key="sel_color")
        fig = px.scatter_map(
            muestra, lat="LAT", lon="LON", color=color_por, color_discrete_map=COLORES[color_por],
            hover_name="Zona",
            hover_data={"Delito": True, "Fecha": "|%d-%m-%Y", "Hora": True, "LAT": False, "LON": False},
            center=CENTRO, zoom=9, opacity=0.5, height=520,
        )
        fig.update_layout(**LAYOUT, map_style="open-street-map")
        st.plotly_chart(fig, width="stretch")

    with col2:
        st.subheader("Mapa de densidad")
        fig = px.density_map(
            muestra, lat="LAT", lon="LON", radius=3, center=CENTRO, zoom=9, height=575,
            color_continuous_scale="YlOrRd", hover_data={"Zona": True},
        )
        fig.update_layout(**LAYOUT, map_style="open-street-map", coloraxis_colorbar_title="Densidad")
        st.plotly_chart(fig, width="stretch")

    st.markdown("**Lectura (versión beta):** los delitos cubren toda la ciudad, pero se concentran en el centro "
                "(Downtown) y el sur (77th Street, Southwest). El Valle de San Fernando y Harbor muestran menor densidad. "
                "Esto sugiere priorizar el patrullaje en el eje centro–sur.")

# ---------- 8.2 CUÁNDO ----------
with tab2:
    col3, col4 = st.columns(2)

    with col3:
        st.subheader("Delitos por mes")
        r_mes = dff.groupby(["Mes", "Gravedad"]).size().reset_index(name="Delitos")
        fig = px.line(r_mes, x="Mes", y="Delitos", color="Gravedad", markers=True,
                      color_discrete_map=COLOR_GRAVEDAD, labels={"Delitos": "Cantidad de delitos"})
        fig.update_layout(**LAYOUT)
        st.plotly_chart(fig, width="stretch")

    with col4:
        st.subheader("Día de la semana × hora")
        fig = px.density_heatmap(dff, x="Hora", y="Día", nbinsx=24, color_continuous_scale="YlOrRd",
                                 category_orders={"Día": ORDEN_DIAS},
                                 labels={"Hora": "Hora del día (0-23 h)", "Día": "Día de la semana"})
        fig.update_layout(**LAYOUT, coloraxis_colorbar_title="Delitos")
        st.plotly_chart(fig, width="stretch")

# ---------- 8.3 QUÉ Y A QUIÉNES ----------
with tab3:
    col5, col6 = st.columns([3, 2])

    with col5:
        st.subheader("Top 10 tipos de delito")
        r_top = dff["Delito"].value_counts().head(10).reset_index()
        r_top.columns = ["Delito", "Delitos"]
        r_top["%"] = (r_top["Delitos"] / len(dff) * 100).round(1)
        fig = px.bar(r_top, x="Delitos", y="Delito", orientation="h", hover_data={"%": True},
                     color_discrete_sequence=["#B9544A"],
                     labels={"Delitos": "Cantidad de delitos", "Delito": "Tipo de delito"})
        fig.update_layout(**LAYOUT, yaxis={"categoryorder": "total ascending"})
        st.plotly_chart(fig, width="stretch")

    with col6:
        st.subheader("Arresto con y sin arma")
        r_arma = (dff.groupby("Arma")["Arresto"].apply(lambda s: (s == "Sí").mean() * 100)
                     .round(1).reset_index(name="% arresto"))
        fig = px.bar(r_arma, x="Arma", y="% arresto", color="Arma", text="% arresto",
                     color_discrete_map=COLOR_ARMA,
                     labels={"Arma": "Uso de arma", "% arresto": "Delitos con arresto (%)"})
        fig.update_layout(**LAYOUT, showlegend=False)
        st.plotly_chart(fig, width="stretch")

    st.subheader("Edad de las víctimas")
    fig = px.histogram(dff[dff["Sexo"] != "Desconocido"], x="Edad", color="Sexo", nbins=40, opacity=0.7,
                       barmode="overlay", color_discrete_map=COLOR_SEXO,
                       labels={"Edad": "Edad de la víctima (años)"})
    fig.update_layout(**LAYOUT, yaxis_title="Cantidad de víctimas")
    st.plotly_chart(fig, width="stretch")

# ── 9. HALLAZGOS PRINCIPALES (coherentes con el Certamen 2) ──
st.divider()
st.subheader("Hallazgos principales")
st.markdown("""
1. **Dónde:** el sur y el centro concentran más delitos (77th Street, Southwest y Central).
2. **Cuándo:** el viernes de 17 a 20 h es el momento más crítico; la madrugada, el más tranquilo.
3. **Qué y a quiénes:** predominan la agresión simple y los robos ligados a vehículos; la mitad de las víctimas tiene entre 27 y 50 años.
4. **Arma y arresto:** los delitos con arma terminan en arresto cerca de 3 veces más que los sin arma.
""")

# ── 10. TABLA DE DATOS (DESPLEGABLE) ─────────────────────────
with st.expander("Ver datos filtrados"):
    columnas = ["Fecha", "Hora", "Zona", "Delito", "Gravedad", "Edad", "Sexo", "Arma", "Arresto"]
    st.dataframe(dff[columnas], width="stretch", hide_index=True)


## 12. Archivos del repositorio
El repositorio debe tener: `crimen_app.py`, `requirements.txt`, `README.md`, la carpeta `.streamlit` con `config.toml` y la carpeta `data` con los datos limpios.

In [ ]:
%%writefile requirements.txt
streamlit
plotly
pandas

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#B9544A"
backgroundColor = "#FBF1EC"
secondaryBackgroundColor = "#F7E1DA"
textColor = "#3A3E48"
font = "sans serif"

In [ ]:
%%writefile README.md
# Crimen en Los Ángeles: ¿dónde, cuándo y qué ocurre?

Dashboard interactivo hecho con **Streamlit** y **Plotly Express** a partir del análisis exploratorio del Certamen 2 de Visualización de la Información (UDD, 2026).

**App publicada:** [PEGAR AQUÍ EL ENLACE .streamlit.app](https://share.streamlit.io)

![Captura del dashboard](captura.png)

## Descripción
El dashboard responde **en qué zonas, en qué momentos y qué tipos de delito concentran la actividad criminal en Los Ángeles entre 2016 y 2019**. Está pensado para quienes planifican el patrullaje y la prevención.

Incluye:
- Filtros globales en el panel lateral: años, zona policial y gravedad del delito.
- KPIs: cantidad de delitos, edad mediana de la víctima, % con arma y % con arresto (comparado con el total).
- Mapa de puntos y mapa de densidad interactivos (`px.scatter_map`, `px.density_map`).
- Gráficos de tipos de delito, edad de las víctimas, evolución mensual, día × hora y relación arma–arresto.

## Datos
- **Fuente:** *Crime Data from 2010 to 2019*, Departamento de Policía de Los Ángeles (LAPD), publicado en Kaggle (licencia CC0).
- **Período:** 2016–2019 (902.673 delitos).
- **Limpieza (Certamen 2):** edades ≤ 0 o > 100 como nulas, coordenadas (0, 0) o fuera de Los Ángeles como nulas, sexo 'H' y '-' como 'X', arma nula como "Sin arma".
- **Muestra:** el archivo completo supera los 25 MB, por lo que `data/crimen_la_limpio.csv` contiene una **muestra aleatoria de 200.000 delitos (`random_state=42`)**.

## Estructura del repositorio
```
crimen_app.py            # la app
requirements.txt         # librerías
README.md
.streamlit/config.toml   # tema de colores
data/crimen_la_limpio.csv
```

## Ejecutar en local
```bash
pip install -r requirements.txt
streamlit run crimen_app.py
```

## Versiones
- `v2.1.0-beta` — Hito 1 (23 de noviembre)
- `v3.0.0` — Examen final (3 de diciembre)

## Integrantes
- Felipe Farías
- Benjamín Godoy


## 13. Probar la app aquí en Colab
Igual que en el tutorial: la primera celda da la contraseña (IP) y la segunda lanza la app con un enlace público `.loca.lt`.

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!npm install -g localtunnel -q

import subprocess, time
subprocess.Popen(["streamlit", "run", "crimen_app.py",
                  "--server.port", "8501",
                  "--server.headless", "true",
                  "--server.enableCORS", "false",
                  "--server.enableXsrfProtection", "false"])
time.sleep(8)
!lt --port 8501

## 14. Descargar los archivos para subirlos a GitHub
Descarga un `.zip` con todo el repositorio listo.

In [ ]:
!zip -r repo_crimen_la.zip crimen_app.py requirements.txt README.md .streamlit data -q
from google.colab import files
files.download("repo_crimen_la.zip")

## 15. Publicar: GitHub → Streamlit Community Cloud
1. **GitHub:** crear un repositorio **público** (ej. `crimen-la-dashboard`) → *Add file → Upload files* → arrastrar el contenido del `.zip` (la carpeta `.streamlit` se puede crear con *Add file → Create new file* escribiendo `.streamlit/config.toml`).
2. **Streamlit Community Cloud** ([share.streamlit.io](https://share.streamlit.io)): *Create app* → elegir el repositorio, rama `main` y archivo `crimen_app.py` → *Deploy*.
3. Copiar el enlace `.streamlit.app` y pegarlo en el `README.md`, junto con una captura de pantalla.
4. **Versión beta:** en GitHub → *Releases → Create a new release* → etiqueta `v2.1.0-beta` → *Publish release*.

### Checklist Hito 1 (versión beta 2.1.0)
| Mínimo | Dónde está |
|---|---|
| Repositorio público y app desplegada | Paso 15 |
| Datos limpios del Certamen 2 | `data/crimen_la_limpio.csv` (sección 1–2) |
| ≥ 2 filtros globales | Años, zonas y gravedad (3) |
| ≥ 3 KPIs | Delitos, edad mediana, % con arma, % con arresto (4) |
| ≥ 3 gráficos Plotly Express | Top delitos, edad, mes, día×hora, arma–arresto (5) |
| Primera versión del mapa | Mapa de puntos + densidad con `px.scatter_map` y `px.density_map` |
| Release `v2.1.0-beta` | Paso 15.4 |

### Pendientes para el Hito 2 (v3.0.0)
| Pendiente | Responsable | Fecha |
|---|---|---|
| Coordenadas paralelas / categorías paralelas (`px.parallel_categories`) | Felipe | 27 nov |
| Sunburst o treemap Zona → Delito (`px.sunburst`) | Benjamín | 27 nov |
| Radar por zona (`px.line_polar`, `line_close=True`) | Felipe | 29 nov |
| Tooltips personalizados en todos los gráficos | Benjamín | 29 nov |
| Interpretación escrita junto a cada mapa y bloque de hallazgos final | Ambos | 1 dic |
| Incorporar retroalimentación del Hito 1, README final y release `v3.0.0` | Ambos | 2 dic |